# Data preprocessing

## Objective

Prepare the cleaned Telco Customer Churn dataset for machine learning using a reproducible and leakage-safe preprocessing workflow.

This notebook will:

- separate the target variable from the predictive features;
- create stratified training and test sets;
- identify numerical, binary, and categorical features;
- handle missing numerical values;
- encode categorical variables;
- preserve preprocessing consistency between training and test data;
- build a reusable `ColumnTransformer` for later modeling.

All preprocessing operations that learn information from the data will be fitted only on the training set to prevent data leakage.

In [143]:
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer

In [144]:
PROJECT_ROOT = Path("..")
DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_PATH = DATA_DIR / "raw" / "WA_Fn-UseC_-Telco-Customer-Churn.csv"
if not RAW_DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found at {RAW_DATA_PATH.resolve()}")
dataset = pd.read_csv(RAW_DATA_PATH)

In [145]:
dataset

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,6840-RESVB,Male,0,Yes,Yes,24,Yes,Yes,DSL,Yes,...,Yes,Yes,Yes,Yes,One year,Yes,Mailed check,84.80,1990.5,No
7039,2234-XADUH,Female,0,Yes,Yes,72,Yes,Yes,Fiber optic,No,...,Yes,No,Yes,Yes,One year,Yes,Credit card (automatic),103.20,7362.9,No
7040,4801-JZAZL,Female,0,Yes,Yes,11,No,No phone service,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.60,346.45,No
7041,8361-LTMKD,Male,1,Yes,No,4,Yes,Yes,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Mailed check,74.40,306.6,Yes


In [146]:
dataset["TotalCharges"] = pd.to_numeric(dataset["TotalCharges"],errors="coerce")

In [147]:
print(dataset["TotalCharges"].dtype)
print("Missing TotalCharges:", dataset["TotalCharges"].isna().sum())

float64
Missing TotalCharges: 11


## Define features and target

The dataset is separated into predictive features (`X`) and the target variable (`y`).

- `X` contains the customer attributes that will be used by the machine learning models to make predictions.
- `y` contains the `Churn` variable, which represents the outcome the models must predict.

The target is encoded as:

- `0`: the customer did not churn;
- `1`: the customer churned.

The `customerID` column is excluded from the predictive features because it is only an identifier and does not represent customer behavior or service characteristics.

In [148]:
X = dataset.drop(columns=["Churn","customerID"])
X

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,Male,0,Yes,Yes,24,Yes,Yes,DSL,Yes,No,Yes,Yes,Yes,Yes,One year,Yes,Mailed check,84.80,1990.50
7039,Female,0,Yes,Yes,72,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,One year,Yes,Credit card (automatic),103.20,7362.90
7040,Female,0,Yes,Yes,11,No,No phone service,DSL,Yes,No,No,No,No,No,Month-to-month,Yes,Electronic check,29.60,346.45
7041,Male,1,Yes,No,4,Yes,Yes,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Mailed check,74.40,306.60


In [149]:
print("Validation...")
print(f"Columns into list: {X.columns.to_list()}")
print(f"Columns len: {len(X.columns.to_list())}")

Validation...
Columns into list: ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges']
Columns len: 19


In [150]:
y = dataset["Churn"].map({'Yes':1,'No':0})
print("Validation...")
print(f"Value counts:\n{y.value_counts()}")
print("-"*50)
print(f"Value counts from dataset:\n{dataset.value_counts("Churn")}")

Validation...
Value counts:
Churn
0    5174
1    1869
Name: count, dtype: int64
--------------------------------------------------
Value counts from dataset:
Churn
No     5174
Yes    1869
Name: count, dtype: int64


In [151]:
assert "customerID" not in X.columns
assert "Churn" not in X.columns
assert X.shape == (7043, 19)

assert y.isna().sum() == 0
assert set(y.unique()) == {0, 1}
assert len(X) == len(y)

print("X and y were created successfully.")

X and y were created successfully.


### Validation results

The feature matrix `X` contains 7,043 customer records and 19 predictive features.

The `customerID` identifier and the `Churn` target were correctly excluded from `X`. The target vector `y` was successfully encoded as:

- `0`: No Churn
- `1`: Churn

The target contains 5,174 customers in class `0` and 1,869 customers in class `1`. No missing values were introduced during the encoding process, and the number of rows in `X` matches the number of observations in `y`.

These validation checks confirm that the features and target were created correctly and are ready for the train-test split.

## Train/test split

The dataset is divided into separate training and test sets before fitting any preprocessing transformations.

- The **training set** contains 80% of the observations and will be used to fit preprocessing steps, train models, perform cross-validation, and select hyperparameters.
- The **test set** contains the remaining 20% and will remain untouched until the final evaluation of the selected model.

A stratified split is used to preserve approximately the same proportion of churned and non-churned customers in both sets. A fixed `random_state` makes the split reproducible.

Creating the split before fitting the preprocessing pipeline prevents data leakage and provides a more reliable estimate of model performance on unseen customers.

In [152]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=.2,random_state=42,stratify=y)

In [153]:
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

print("-"*50)

print("\nTraining target proportions:")
display(y_train.value_counts(normalize=True).round(4)*100)
print("Test target proportions:")
display(y_test.value_counts(normalize=True).round(4)*100)

X_train shape: (5634, 19)
X_test shape: (1409, 19)
y_train shape: (5634,)
y_test shape: (1409,)
--------------------------------------------------

Training target proportions:


Churn
0    73.46
1    26.54
Name: proportion, dtype: float64

Test target proportions:


Churn
0    73.46
1    26.54
Name: proportion, dtype: float64

### Split validation

The dataset was successfully divided into:

- **Training set:** 5,634 customers and 19 predictor features.
- **Test set:** 1,409 customers and 19 predictor features.

The target arrays contain the corresponding 5,634 training labels and 1,409 test labels, confirming that each observation has a target value.

Both sets preserve the original target distribution:

- approximately **73.5%** of customers did not churn (`0`);
- approximately **26.5%** of customers churned (`1`).

Therefore, `stratify=y` worked as intended. The training and test sets have comparable class distributions, reducing the possibility that model evaluation will be distorted by an unrepresentative split.

The test set will now remain untouched until the final evaluation of the selected model. All preprocessing transformations will be fitted using only the training data.

## Feature type identification

The predictor features are separated into numerical and categorical groups because they require different preprocessing transformations.

- **Numerical features** contain quantitative values and will be imputed and standardized when required.
- **Categorical features** contain nominal categories and will be imputed and converted into numerical columns using one-hot encoding.

Feature types are identified using only `X_train`. This keeps the preprocessing workflow based exclusively on the training data and helps prevent accidental use of the test set.

In [154]:
PROJECT_ROOT = Path("..")
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DATA_PATH = DATA_DIR / "processed" / "column_audit.csv"
if not PROCESSED_DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found at {PROCESSED_DATA_PATH.resolve()}")
column_audit = pd.read_csv(PROCESSED_DATA_PATH, index_col=0)
column_audit

,column,dtype,unique_count,possible_role
0,customerID,str,7043,identifier
1,gender,str,2,binary categorical
2,SeniorCitizen,int64,2,binary categorical stored as integer
3,Partner,str,2,binary categorical
4,Dependents,str,2,binary categorical
5,tenure,int64,73,numeric
6,PhoneService,str,2,binary categorical
7,MultipleLines,str,3,multiclass categorical
8,InternetService,str,3,multiclass categorical
9,OnlineSecurity,str,3,multiclass categorical


In [155]:
numerical_features = ['tenure', 'MonthlyCharges', 'TotalCharges']
binary_features = ["SeniorCitizen"]
categorical_features = X_train.select_dtypes(include="str").columns.to_list()

In [156]:
all_classified_features = (numerical_features + binary_features + categorical_features)

assert len(all_classified_features) == X_train.shape[1]
assert set(all_classified_features) == set(X_train.columns)

print("Numerical:", numerical_features)
print("Binary:", binary_features)
print("Categorical:", categorical_features)
print("\nAll features were classified successfully.")

Numerical: ['tenure', 'MonthlyCharges', 'TotalCharges']
Binary: ['SeniorCitizen']
Categorical: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']

All features were classified successfully.


In [157]:
column_audit = column_audit[(column_audit["column"] != "customerID") & (column_audit["column"] != "Churn")]
column_audit.loc[column_audit["column"] == "TotalCharges", "dtype"] = "float64"

In [158]:
binary_feature_preprocess = column_audit[(column_audit["unique_count"] == 2) & (column_audit["dtype"] != "int64")]
binary_feature_preprocess = binary_feature_preprocess["column"].to_list()

multiclass_feature_preprocess = column_audit[
    (column_audit["unique_count"] > 2) &
    (column_audit["dtype"] != "int64") &
    (column_audit["dtype"] != "float64")
]
multiclass_feature_preprocess = multiclass_feature_preprocess["column"].to_list()

print(f'Binary categorizal:\n{binary_feature_preprocess}')
print(f'Multicass categorical:\n{multiclass_feature_preprocess}')

Binary categorizal:
['gender', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']
Multicass categorical:
['MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaymentMethod']


In [159]:
if len(categorical_features) == len(multiclass_feature_preprocess + binary_feature_preprocess):
    print(f"All features balanced")
else:
    print(f"All features desbalanced")

All features balanced


## Preprocessing pipeline construction

A reusable preprocessing workflow is created to apply the appropriate transformation to each feature group.

The transformations are defined as follows:

- **Numerical features:** missing values are replaced with `0`, and the resulting values are standardized.
- **Categorical features:** categories are converted into numerical columns using one-hot encoding.
- **Binary numerical features:** `SeniorCitizen` is already represented as `0` and `1`, so it does not require encoding and can be passed through unchanged.

The individual transformations are combined with a `ColumnTransformer`. This allows each group of columns to receive its corresponding preprocessing steps while preserving a single reproducible workflow.

The preprocessor will later be fitted only on the training data and applied to the test data without refitting, preventing data leakage.

In [160]:
numerical_pipeline = Pipeline(
    steps=[
        ("imputer",SimpleImputer(strategy="constant",fill_value=0)),
        ("scaler",StandardScaler())
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

In [161]:
preprocessor = ColumnTransformer(
    transformers=[
        ("numerical_pipeline", numerical_pipeline, numerical_features),
        ("categorical_pipeline", categorical_pipeline, (binary_feature_preprocess + multiclass_feature_preprocess))
    ]
)

In [162]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

In [164]:
X_train_processed_df = pd.DataFrame(X_train_processed)
X_train_processed_df

,0,1,2,3,4,5,6,7,8,9,...,34,35,36,37,38,39,40,41,42,43
0,0.102371,-0.521976,-0.262257,0.0,1.0,1.0,0.0,1.0,0.0,1.0,...,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
1,-0.711743,0.337478,-0.503635,0.0,1.0,0.0,1.0,0.0,1.0,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
2,-0.793155,-0.809013,-0.749883,0.0,1.0,0.0,1.0,0.0,1.0,1.0,...,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
3,-0.263980,0.284384,-0.172722,1.0,0.0,0.0,1.0,1.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
4,-1.281624,-0.676279,-0.989374,0.0,1.0,0.0,1.0,0.0,1.0,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5629,1.567778,1.470695,2.373129,1.0,0.0,0.0,1.0,1.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
5630,-1.240918,-0.626504,-0.973665,0.0,1.0,1.0,0.0,1.0,0.0,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
5631,-0.304686,1.256662,0.158344,1.0,0.0,1.0,0.0,1.0,0.0,0.0,...,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
5632,-0.345392,-1.477661,-0.797075,1.0,0.0,0.0,1.0,1.0,0.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0


### Preprocessing pipeline interpretation

The preprocessing workflow was successfully organized with a `ColumnTransformer`.

The numerical pipeline replaces missing values with `0` and standardizes the numerical variables. The `SeniorCitizen` feature is preserved without modification because it is already encoded as a binary numerical variable. All string-based categorical features are transformed using one-hot encoding.

Using `handle_unknown="ignore"` ensures that categories encountered outside the training data do not cause an error during transformation.

At this stage, the preprocessing structure has only been defined. It has not yet learned information from the data. The next step is to fit the preprocessor on `X_train` and use the fitted transformation on both the training and test sets.